In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, random_split, DataLoader

In [2]:
#import comet_ml
#from comet_ml import Experiment
#COMET_API_KEY = ""

#assert COMET_API_KEY != ""

In [ ]:
if torch.cuda.is_available():
    print("CUDA is available!")
    device = torch.device('cuda')
    
else:
    print("CUDA is not available!")
    device = torch.device('cpu')
    

In [4]:
import pandas
import os
import pickle as pkl
import itertools

In [5]:
# downloading the dataset to the right folder
DATASETS_PATH = os.path.join('..','..','..','data')
TEST_DATASET_PATH = os.path.join(DATASETS_PATH, 'test.pickle')
TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, 'train.pickle')

In [ ]:
# opening test and train dataset with pickle - the resulting object is a pandas.DataFrame
with open(TEST_DATASET_PATH, 'rb') as f:
    test_dataset = pkl.load(f)

with open(TRAIN_DATASET_PATH, 'rb') as f:
    train_dataset = pkl.load(f)

type(test_dataset)

In [7]:
    
class SensorDataSet(Dataset): 
    def __init__(self, dataset: pandas.DataFrame):
        self.dataset = dataset

    def __getitem__(self, idx):
        data = self.dataset.iloc[idx] 
        return torch.tensor(data['sensor_data'], dtype=torch.float32), data['label']

    def __len__(self):
        return len(self.dataset.index)



# use pytorch's dataloader
from torch.utils.data import DataLoader

test_dataloader1 = DataLoader(SensorDataSet(test_dataset), batch_size=32)
train_dataloader1 = DataLoader(SensorDataSet(train_dataset), batch_size=32)


In [8]:
# Wrapping the full DataFrame in the dataset class
full_dataset = SensorDataSet(train_dataset)

# Defining split sizes
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size


In [9]:
#def create_experiment(name, params):
#    experiment = Experiment(
#        api_key="p7oQkh1TGxiQ87LVp7kUQevDP",  
#        project_name="fcn-generative-example",
#        workspace="helmi0000"
#    )
#    experiment.set_name(name)
#    experiment.log_parameters(params)
#    return experiment

In [10]:
def build_fc_model():
    fc_model = nn.Sequential(
        nn.Flatten(),

        nn.Linear(128 * 6, 128),
        
        nn.ReLU(),
        
        nn.Linear(128, 64),
        
        nn.ReLU(),
        
        nn.Linear(64, 12)
    )
    return fc_model

fc_model_sequential = build_fc_model()

In [11]:
from model import FullyConnectedModel
    
fc_model = FullyConnectedModel().to(device)

In [12]:
loss_function = nn.CrossEntropyLoss()

In [13]:
# for the comet experiment add the parameter , experiment to the input

def train(model, train_dataloader, val_dataloader, criterion, optimizer, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct_pred = 0
        total_pred = 0

        for images, labels in train_dataloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)
            predicted = torch.argmax(outputs, dim=1)
            correct_pred += (predicted == labels).sum().item()
            total_pred += labels.size(0)

        total_epoch_loss = total_loss / total_pred
        epoch_accuracy = correct_pred / total_pred
        print(f"Epoch {epoch + 1}, Train Loss: {total_epoch_loss:.4f}, Train Accuracy: {epoch_accuracy:.4f}")
        #experiment.log_metric("train_loss", total_epoch_loss, step=epoch, epoch=epoch)
        #experiment.log_metric("train_accuracy", epoch_accuracy, step=epoch, epoch=epoch) 

        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_dataloader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                predicted = torch.argmax(outputs, dim=1)
                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)

        val_epoch_loss = val_loss / val_total
        val_epoch_accuracy = val_correct / val_total
        print(f"Epoch {epoch + 1}, Randsplit Eval Loss: {val_epoch_loss:.4f}, Randsplit Eval Accuracy: {val_epoch_accuracy:.4f}")
        #experiment.log_metric("Randomsplit Eval Loss", val_epoch_loss, step=epoch, epoch=epoch)
        #experiment.log_metric("Randomsplit Eval Accuracy", val_epoch_accuracy, step=epoch, epoch=epoch)

In [14]:
# for comet experiment add the input , experiment to evaluate
def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            predicted = torch.argmax(outputs, dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    avg_loss = total_loss / total
    accuracy = correct / total
    #experiment.log_metric("test_loss", avg_loss)
    #experiment.log_metric("test_accuracy", accuracy)
    return avg_loss, accuracy

In [15]:
optimizers = ["SGD", "Adam","RMSprop", "Adagrad"]
learning_rates = [0.001 , 0.005 , 0.01 , 0.015 ]
epochs_list = [7]
batch_sizes = [32, 64]


In [ ]:
# Run experiments for all 32 combinations 

for optimizer_name, lr, epochs, batch_size in itertools.product(optimizers, learning_rates, epochs_list, batch_sizes):
    name = f"{optimizer_name}_lr={lr}_ep={epochs}_bs={batch_size}"
    params = {
        "optimizer": optimizer_name,
        "learning_rate": lr,
        "epochs": epochs,
        "batch_size": batch_size
    }
    print(f"\n Running Experiment: {name}")

    # experiment = create_experiment(name, params)

    split_accuracies = []
    # I used the index split_id as the random seed so that I can have the same random values each time I change the parameters
    for split_id in range(10):
        print(f"Split {split_id + 1}/10 for {name}")

        train_subset, val_subset = random_split(
            full_dataset,
            [train_size, val_size],
            generator=torch.Generator().manual_seed(split_id)
        )
        train_dataloader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
        val_dataloader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

        model = FullyConnectedModel().to(device)
        criterion = nn.CrossEntropyLoss()
        if optimizer_name == "SGD":
           optimizer = torch.optim.SGD(model.parameters(), lr=lr)
        elif optimizer_name == "Adam":
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        elif optimizer_name == "RMSprop":
            optimizer = torch.optim.RMSprop(model.parameters(), lr=lr)
        elif optimizer_name == "Adagrad":
            optimizer = torch.optim.Adagrad(model.parameters(), lr=lr)
        train(model, train_dataloader, val_dataloader, criterion, optimizer, epochs)
        # for comet experiment add the parameter , experiment
        _, accuracy = evaluate(model, test_dataloader1, criterion)
        # for comet experiment add the parameter , experiment , experiment
        
    #experiment.end()